# Plan & Execute — 스스로 계획하고 실행하는 Agent

**Plan-and-Execute** 는 복잡한 목표를 먼저 **단계별 계획(plan)** 으로 쪼갠 뒤, 각 단계를 실행하고, 남은 상황을 보고 **계획을 갱신(replan)** 하는 과정을 반복하는 패턴이다.

한 번에 다 풀려는 ReAct 식 접근보다, 긴 호흡의 문제에서 **명시적 계획** 덕에 길을 잃지 않는 장점이 있다.

```
START → planner → agent(실행) → replan ──(끝?)──▶ END
                    ▲                  │
                    └──────────────────┘ (남은 단계 있으면 다시 실행)
```

구성: **Planner**(계획 수립) · **Executor**(단계 실행, ReAct 에이전트) · **RePlanner**(계획 갱신/종료 판단).

## 환경 변수 준비

프로젝트 루트의 `.env` 에 키를 넣고 `load_dotenv()` 로 불러온다.

```
OPENAI_API_KEY=sk-...
TAVILY_API_KEY=tvly-...     # 웹검색이 필요한 노트북
ANTHROPIC_API_KEY=sk-ant-... # Claude 를 쓰는 노트북
```

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

assert os.environ.get("OPENAI_API_KEY"), "OPENAI_API_KEY 가 .env 에 없습니다"
assert os.environ.get("TAVILY_API_KEY"), "TAVILY_API_KEY 가 .env 에 없습니다"
print("환경변수 로드 완료:", "OPENAI_API_KEY", "TAVILY_API_KEY")

## Step 1. 실행기(Executor) — 웹검색 ReAct 에이전트

[basics 복습] `create_react_agent` prebuilt 로 "도구를 부르며 한 단계를 수행"하는 실행기를 만든다. 여기선 Tavily 웹검색 도구를 쥐여준다.

In [ ]:
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_openai import ChatOpenAI
from langgraph.prebuilt import create_react_agent

search_tool = TavilySearchResults(max_results=3)
tools = [search_tool]

llm = ChatOpenAI(model="gpt-4o")
plan_executor = create_react_agent(llm, tools, prompt="You are a helpful assistant.")

In [ ]:
# 단독 동작 확인
result = plan_executor.invoke({"messages": [("user", "2025년 한국의 최저시급은 얼마입니까?")]})
print(result["messages"][-1].content)

## Step 2. Planner — 목표를 단계별 계획으로

[basics 복습] `with_structured_output(Plan)` 으로 LLM 이 **단계 리스트** 를 정해진 형식으로 내게 한다.

In [ ]:
from pydantic import BaseModel, Field
from typing import List
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage

class Plan(BaseModel):
    """향후 따라야 할 계획"""
    steps: List[str] = Field(description="수행할 단계들 (정렬된 순서)")

planner_prompt = ChatPromptTemplate.from_messages([
    ("system",
     """주어진 목표에 대해 간단한 단계별 계획을 세우세요. \
     각 단계는 올바르게 실행되면 정답으로 이어지는 개별 작업이어야 합니다. \
     불필요한 단계는 넣지 마세요. 마지막 단계의 결과가 최종 답이어야 합니다. \
     각 단계에 필요한 정보가 모두 담기게 하고, 단계를 건너뛰지 마세요."""),
    MessagesPlaceholder(variable_name="messages"),
])
planner = planner_prompt | llm.with_structured_output(Plan)

In [ ]:
plan_result = planner.invoke(
    {"messages": [HumanMessage(content="2025년 한국에서 개봉한 영화 중 가장 흥행한 영화는?")]}
)
plan_result.steps

## Step 3. RePlanner — 계획 갱신 또는 종료 판단

이미 수행한 단계를 보고, ① 더 진행할 단계가 있으면 **갱신된 계획**(`Plan`)을, ② 답할 수 있으면 **최종 응답**(`Response`)을 낸다. `Union` 으로 둘 중 하나를 고르게 한다.

In [ ]:
from typing import Union

replanner_prompt = ChatPromptTemplate.from_template(
    """주어진 목표에 대해 간단한 단계별 계획을 세우세요. 불필요한 단계는 넣지 마세요.

    목표: {input}
    원래 계획: {plan}
    완료한 단계들: {past_steps}

    이에 맞춰 계획을 갱신하세요. 더 이상 단계가 필요 없고 사용자에게 답할 수 있으면 그렇게 응답하세요.
    그렇지 않으면 아직 '해야 할' 단계만 계획에 채우세요. 이미 완료한 단계는 다시 넣지 마세요."""
)

class Response(BaseModel):
    """사용자에게 줄 최종 응답"""
    response: str

class Act(BaseModel):
    """수행할 행동"""
    action: Union[Response, Plan] = Field(
        description="사용자에게 답하려면 Response, 도구로 더 알아봐야 하면 Plan 을 사용"
    )

replanner = replanner_prompt | llm.with_structured_output(Act)

## Step 4. 그래프 생성

[basics 복습] State 에 `past_steps` 는 `operator.add` 리듀서로 완료 단계를 **누적** 한다.

In [ ]:
import operator
from typing import Annotated, List, Tuple
from typing_extensions import TypedDict

class PlanExecute(TypedDict):
    input: str
    plan: List[str]
    past_steps: Annotated[List[Tuple], operator.add]   # 완료 단계 누적
    response: str

In [ ]:
# 계획 생성 노드
def plan_step(state: PlanExecute):
    plan = planner.invoke({"messages": [("user", state["input"])]})
    return {"plan": plan.steps}

# 계획 실행 노드 (계획의 첫 단계를 실행기로 수행)
def execute_step(state: PlanExecute):
    plan = state["plan"]
    plan_str = "\n".join(f"{i+1}. {step}" for i, step in enumerate(plan))
    task = plan[0]
    task_formatted = f"다음 계획에 대해:\n{plan_str}\n\n당신은 1단계 '{task}' 를 실행합니다."
    agent_response = plan_executor.invoke({"messages": [("user", task_formatted)]})
    return {"past_steps": [(task, agent_response["messages"][-1].content)]}

# 재계획 노드
def replan_step(state: PlanExecute):
    output = replanner.invoke(state)
    if isinstance(output.action, Response):   # 바로 답 가능
        return {"response": output.action.response}
    return {"plan": output.action.steps}      # 아직 할 단계 남음

In [ ]:
from langgraph.graph import StateGraph, START, END

# response 가 채워졌으면 종료, 아니면 다시 실행기로
def should_end(state: PlanExecute):
    if state.get("response"):
        return END
    return "agent"

gb = StateGraph(PlanExecute)
gb.add_node("planner", plan_step)
gb.add_node("agent", execute_step)
gb.add_node("replan", replan_step)
gb.add_edge(START, "planner")
gb.add_edge("planner", "agent")
gb.add_edge("agent", "replan")
gb.add_conditional_edges("replan", should_end, ["agent", END])
graph = gb.compile()

In [ ]:
from IPython.display import Image, display

try:
    display(Image(graph.get_graph(xray=True).draw_mermaid_png()))
except Exception:
    print(graph.get_graph().draw_mermaid())

실행 — 계획 수립 → 단계 실행 → 재계획 이 반복되다 답이 나오면 종료한다. `recursion_limit` 으로 과도한 반복을 막는다.

In [ ]:
config = {"recursion_limit": 50}
inputs = {"input": "2024년 노벨문학상 수상자의 출신 국가는 어디인가요?"}

for event in graph.stream(inputs, config=config, stream_mode="values"):
    for k, v in event.items():
        print(f"[{k}]", v)
    print("=" * 40)

## 정리

- **Plan & Execute** = 계획 수립 → 단계 실행 → 재계획의 반복
- **Planner**: `with_structured_output(Plan)` 로 단계 리스트 생성
- **Executor**: `create_react_agent` 로 각 단계를 도구와 함께 수행
- **RePlanner**: `Union[Response, Plan]` 로 '종료 vs 계속' 을 판단
- `past_steps` 를 리듀서로 누적해 진행 상황을 추적

다음: 코드를 생성·실행·검증하며 에러를 스스로 고치는 데이터 분석 Agent.